In [1]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
import m2cgen as m2c
import numpy as np
from collections import deque
from sklearn.model_selection import train_test_split

In [7]:
df1=pd.read_csv('C:/Users/bogda/gazpromhack/data/matan25.csv', encoding='UTF-8', sep=',')
df1.head()
df1=df1[380:]
df1

,Unnamed: 0,TimeStamp,Температура масла в магистрали общей откачки (поз. Тм),Температура слива масла из опоры турбины (поз. Т606),Температура масла на входе в двигатель за фильтром (поз. Т607),Температура топливного газа перед СК (перед расходомерным устройством) (поз. Ттг),Давление масла в нагнетающей магистрали двигателя (после фильтра) (поз. Pм),Давление масла в магистрали общей откачки (поз. Р615),Давление за ОК Pk1,Давление за ОК Pk2,...,Частота вращения РНД (Расчетная),Частота вращения РВД (Расчетная),Частота вращения РСТ (Расчетная),Температура воздуха на входе в газогенератор (Расчетная),Давление воздуха за компрессором (Расчетная),Частота вращения РНД приведенная,Частота вращения РВД приведенная,Температура газов за ТНД максимальная (Расчетная),Положение лопаток НА КВД. (поз. A2),Положение КПВ (поз. A3)
380,380,8739,21.548,28.160,0.220,0.220000,41.880747,122.904630,244.713,257.590,...,4052.990,9187.356964,1490.427,221.797334,250.959,4139.992893,9722.291,439.457,0.220,8.072
381,381,8764,21.548,28.160,0.220,0.220000,455.497173,351.949900,244.713,257.590,...,4052.990,9187.356964,1490.427,221.797334,250.959,4019.434217,9722.291,439.457,0.220,8.072
382,382,8790,21.548,28.160,0.220,0.220000,424.369877,454.033117,244.713,257.590,...,4052.990,9187.356964,1490.427,221.797334,250.959,4321.860132,9722.291,439.470,0.220,8.072
383,383,8808,21.548,28.160,0.220,106.732005,252.255106,146.785689,244.713,257.590,...,4052.990,9187.356964,1490.427,221.797334,250.959,4293.856980,9722.291,439.470,0.220,8.072
384,384,8831,21.548,28.160,0.220,0.220000,465.123337,242.948006,244.713,257.590,...,4052.990,9187.356964,1490.427,221.797334,250.959,4283.648129,9722.291,439.470,0.220,8.072
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12654,12654,51044,43.561,65.943,23.177,23.477000,308.473000,199.926000,935.612,941.988,...,7699.703,10666.571000,3697.968,4.822000,939.183,7834.907000,10856.534,586.625,79.420,9.020
12655,12655,51064,43.561,65.943,23.107,23.477000,308.473000,199.926000,933.762,942.065,...,7700.150,10667.598000,3698.198,4.822000,938.800,7837.355000,10857.264,586.625,79.667,9.014
12656,12656,51087,43.561,65.943,23.107,23.477000,308.473000,199.926000,933.762,942.065,...,7701.266,10666.765000,3698.287,4.822000,937.913,7837.810000,10858.309,586.625,79.667,9.014
12657,12657,51111,43.561,65.943,23.107,23.477000,308.232000,200.667000,934.841,942.065,...,7701.266,10666.765000,3698.287,4.822000,937.913,7838.946000,10857.461,586.625,79.637,8.975


In [2]:
df=pd.read_csv('C:/Users/bogda/gazpromhack/data/1.csv', encoding='cp1251', sep=';')
df=df[380:]
df.head()

,TimeStamp,Температура масла в магистрали общей откачки (поз. Тм),Температура слива масла из опоры турбины (поз. Т606),Температура масла на входе в двигатель за фильтром (поз. Т607),Температура топливного газа перед СК (перед расходомерным устройством) (поз. Ттг),Давление масла в нагнетающей магистрали двигателя (после фильтра) (поз. Pм),Давление масла в магистрали общей откачки (поз. Р615),Давление за ОК Pk1,Давление за ОК Pk2,Давление топливного газа перед дозаторами (поз. Pтг1.1),...,Частота вращения РНД (Расчетная),Частота вращения РВД (Расчетная),Частота вращения РСТ (Расчетная),Температура воздуха на входе в газогенератор (Расчетная),Давление воздуха за компрессором (Расчетная),Частота вращения РНД приведенная,Частота вращения РВД приведенная,Температура газов за ТНД максимальная (Расчетная),Положение лопаток НА КВД. (поз. A2),Положение КПВ (поз. A3)
380,12:10:08.739,22.176,29.949,24.431,9.069,302.039,199.092,245.792,258.740,2915.428,...,4067.290,9565.749,1495.981,5.485,252.266,4133.713,9725.538,439.457,16.301,8.170
381,12:10:08.764,22.176,29.949,24.431,9.069,302.392,196.403,245.869,257.361,2917.776,...,4067.290,9565.749,1495.981,5.485,252.266,4135.080,9725.182,439.457,16.387,8.302
382,12:10:08.790,22.176,29.949,24.431,9.069,302.392,196.403,245.869,257.361,2917.776,...,4069.346,9565.982,1495.985,5.485,251.615,4135.080,9725.182,439.470,16.387,8.302
383,12:10:08.808,22.176,29.949,24.431,9.069,302.318,192.733,245.869,259.659,2917.776,...,4069.346,9565.982,1495.985,5.485,251.615,4137.169,9725.419,439.470,16.387,8.229
384,12:10:08.831,22.176,29.949,24.431,9.069,302.243,189.507,244.790,258.816,2915.304,...,4067.718,9565.975,1495.984,5.485,252.764,4137.169,9725.419,439.470,16.365,8.260


In [22]:
df3=pd.read_csv('C:/Users/bogda/gazpromhack/data/1.csv', encoding='cp1251', sep=';').loc[:, ['TimeStamp', 'Положение КПВ (поз. A3)']]
df3.to_csv('PositionKPV.csv', index=False)

In [ ]:
class EnsembleDetector:
    def __init__(self, slew_threshold, std_threshold, kalman_threshold, 
                 window_size=20, persistence=2, k_q_val=0.0001, k_r_val=0.1):
        self.slew_threshold = slew_threshold
        self.last_val = None
        
        self.window_size = int(window_size)
        self.buffer = np.zeros(self.window_size)
        self.buf_idx = 0
        self.count = 0
        self.std_threshold = std_threshold
        
        self.k_x = 0.0      
        self.k_v = 0.0      
        self.k_p = np.eye(2) 
        self.k_q = np.eye(2) * k_q_val 
        self.k_r = k_r_val      
        self.kalman_threshold = kalman_threshold
        self.is_kf_init = False

        self.persistence_limit = persistence
        self.suspect_counter = 0

    def _kalman_step(self, val):
        if not self.is_kf_init:
            self.k_x = val
            self.is_kf_init = True
            return 0.0
        
        x_pred = self.k_x + self.k_v
        v_pred = self.k_v
        self.k_p[0,0] += self.k_p[1,1] + self.k_q[0,0]
        
        innovation = val - x_pred
        s = self.k_p[0,0] + self.k_r
        k_gain_x = self.k_p[0,0] / s
        k_gain_v = self.k_p[0,1] / s

        self.k_x = x_pred + k_gain_x * innovation
        self.k_v = v_pred + k_gain_v * innovation
        self.k_p[0,0] *= (1 - k_gain_x)
        
        return abs(innovation)

    def update(self, val):
        slew_err = abs(val - self.last_val) if self.last_val is not None else 0
        slew_alarm = slew_err > self.slew_threshold
        
        self.buffer[self.buf_idx] = val
        self.buf_idx = (self.buf_idx + 1) % self.window_size
        self.count = min(self.count + 1, self.window_size)
        current_std = np.std(self.buffer[:self.count]) if self.count > 1 else 0
        std_alarm = current_std > self.std_threshold
        
        kalman_err = self._kalman_step(val)
        kalman_alarm = kalman_err > self.kalman_threshold

        if slew_alarm or kalman_alarm or std_alarm:
            self.suspect_counter += 1
        else:
            self.suspect_counter = 0

        final_alarm = (self.suspect_counter >= 1) and (self.suspect_counter <= self.persistence_limit)
        
        self.last_val = val
        return {
            'val': val,
            'slew_err': slew_err,
            'std_val': current_std,
            'kalman_err': kalman_err,
            'alarm': final_alarm
        }

In [ ]:
class RobustSpikeDetector:
    def __init__(self, slew_min=1.0, k_factor=4.0, max_spike_duration=3):
        self.slew_min = slew_min
        self.k_factor = k_factor
        self.max_duration = max_spike_duration
        
        self.last_val = None
        self.anomaly_counter = 0
        
        self.k_x = 0.0
        self.k_p = 1.0
        self.k_q = 0.01 
        self.k_r = 0.1
        self.is_init = False

    def update(self, val):
        if not self.is_init:
            self.k_x = val
            self.last_val = val
            self.is_init = True
            return False, 0.0

        prediction_err = abs(val - self.k_x)
        
        slew_rate = abs(val - self.last_val)
        
        is_suspicious = (slew_rate > self.slew_min) and (prediction_err > (self.k_factor * self.k_r))
        
        if is_suspicious:
            self.anomaly_counter += 1
        else:
            self.anomaly_counter = 0

        is_alarm = 0 < self.anomaly_counter <= self.max_duration
        
        self.k_p += self.k_q
        k_gain = self.k_p / (self.k_p + self.k_r)
        self.k_x = self.k_x + k_gain * (val - self.k_x)
        self.k_p *= (1 - k_gain)
        
        self.last_val = val
        return is_alarm, slew_rate

In [ ]:
class TrendingCusumDetectorFiltered:
    def __init__(self, target_slope, sigma_slope, h_factor=50, k_factor=2.0, alpha=0.2):
        self.target_slope = target_slope
        
        self.K = k_factor * sigma_slope 
        
        self.H = h_factor * sigma_slope
        
        self.s_high = 0.0
        self.s_low = 0.0
        
        self.last_val = None
        self.smoothed_val = None
        self.alpha = alpha
        
        self.decay = 0.99 

    def update(self, val):
        if self.smoothed_val is None:
            self.smoothed_val = val
            self.last_val = val
            return False, 0, 0
        
        self.smoothed_val = self.alpha * val + (1 - self.alpha) * self.smoothed_val
        current_slope = self.smoothed_val - self.last_val
        self.last_val = self.smoothed_val
        
        diff = current_slope - self.target_slope
        
        self.s_high = max(0, (self.s_high + diff - self.K) * self.decay)
        self.s_low = max(0, (self.s_low - diff - self.K) * self.decay)
        
        alarm = (self.s_high > self.H) or (self.s_low > self.H)
        return alarm, self.s_high, self.s_low

In [ ]:
def analyze_sensor_data(df, sensor_column, alpha=0.1):
    data = df[sensor_column].values
    
    train_idx = min(400, len(data))
    training_data = data[:train_idx]
    
    smoothed_train = []
    curr = training_data[0]
    for v in training_data:
        curr = alpha * v + (1 - alpha) * curr
        smoothed_train.append(curr)
    
    deltas = np.diff(smoothed_train)
    target_slope = np.mean(deltas)
    sigma_slope = np.std(deltas)
    
    if sigma_slope < 1e-6: sigma_slope = 1e-6

    detector = TrendingCusumDetectorFiltered(
        target_slope=target_slope, 
        sigma_slope=sigma_slope, 
        h_factor=100, 
        k_factor=5.0, 
        alpha=alpha
    )

    results = []
    for val in data:
        alarm, sh, sl = detector.update(val)
        results.append({'val': val, 's_high': sh, 's_low': sl, 'alarm': alarm})
    
    return pd.DataFrame(results)

d=analyze_sensor_data(df, 'Положение КПВ (поз. A3)')
d['alarm'].value_counts()

d3=analyze_sensor_data(df1, 'Положение КПВ (поз. A3)')
d3['alarm'].value_counts()

alarm
True     11869
False      410
Name: count, dtype: int64

In [ ]:
def analyze_ensemble(df, col, slew_factor=3, std_factor=3, k_factor=3, p=3):
    data = df[col].ffill().bfill().values
    train = data[:100]
    
    s_th = np.median(np.abs(np.diff(train))) * slew_factor 
    std_th = np.std(train) * std_factor
    
    kf_temp = EnsembleDetector(1000, 1000, 1000, 20, 5, 1e-4, 1e-2)
    k_errors = [kf_temp.update(v)['kalman_err'] for v in train]
    k_th = (np.mean(k_errors) + np.std(k_errors)) * k_factor
    
    detector = EnsembleDetector(
        slew_threshold=s_th, 
        std_threshold=std_th, 
        kalman_threshold=k_th, 
        window_size=20, 
        persistence=p,
        k_q_val=1e-4, 
        k_r_val=1e-2
    )
    
    results = [detector.update(v) for v in data]
    res_df = pd.DataFrame(results)
    return res_df

def analyze_final(df, col, slew_min=2.1, k_f=5.0, max_dur=3):
    data = df[col].ffill().values
    detector = RobustSpikeDetector(slew_min=slew_min, k_factor=k_f, max_spike_duration=max_dur)
    
    results = []
    for v in data:
        alarm, slew = detector.update(v)
        results.append({'val': v, 'slew': slew, 'alarm1': alarm})
        
    return pd.DataFrame(results)

d1=analyze_final(df,'Температура газов за ТНД максимальная (Расчетная)')
d1['alarm1'].value_counts()

d2=analyze_final(df1,'Температура газов за ТНД максимальная (Расчетная)')
d2['alarm1'].value_counts()

alarm1
False    11938
True       341
Name: count, dtype: int64

In [9]:
d

,val,s_high,s_low,alarm
0,8.170,0,0.0,False
1,8.302,0,0.0,False
2,8.302,0,0.0,False
3,8.229,0,0.0,False
4,8.260,0,0.0,False
...,...,...,...,...
12274,9.020,0,0.0,False
12275,9.014,0,0.0,False
12276,9.014,0,0.0,False
12277,8.975,0,0.0,False


In [18]:
d3

,val,s_high,s_low,alarm
0,8.072,0.0,0.000000,False
1,8.072,0.0,0.000000,False
2,8.072,0.0,0.000000,False
3,8.072,0.0,0.000000,False
4,8.072,0.0,0.000000,False
...,...,...,...,...
12274,9.020,0.0,62.314279,True
12275,9.014,0.0,62.844365,True
12276,9.014,0.0,63.253826,True
12277,8.975,0.0,63.559262,True


In [43]:
dataset_temp = pd.concat([d, d1], axis=1)
dataset_temp['alarm_itog'] = dataset_temp['alarm'] & dataset_temp['alarm1']
dataset_temp

,val,s_high,s_low,alarm,val,slew,alarm1,alarm_itog
0,439.457,0.0,0.0,False,439.457,0.000,False,False
1,439.457,0.0,0.0,False,439.457,0.000,False,False
2,439.470,0.0,0.0,False,439.470,0.013,False,False
3,439.470,0.0,0.0,False,439.470,0.000,False,False
4,439.470,0.0,0.0,False,439.470,0.000,False,False
...,...,...,...,...,...,...,...,...
12274,586.625,0.0,0.0,False,586.625,0.000,False,False
12275,586.625,0.0,0.0,False,586.625,0.000,False,False
12276,586.625,0.0,0.0,False,586.625,0.000,False,False
12277,586.625,0.0,0.0,False,586.625,0.000,False,False


In [52]:
dataset_temp1 = pd.concat([d2, d3], axis=1)
dataset_temp1['alarm_itog'] = dataset_temp1['alarm'] & dataset_temp1['alarm1']
dataset_temp1['alarm_itog'].value_counts()
dataset_temp1=dataset_temp1.drop('alarm',axis=1)
dataset_temp1=dataset_temp1.drop('alarm1',axis=1)
dataset_temp1.columns = ['val', 'slew', 'val_2', 's_high', 's_low', 'alarm_itog']
dataset_temp1=dataset_temp1.drop('val_2',axis=1)
dataset_temp1

,val,slew,s_high,s_low,alarm_itog
0,439.457,0.000,0.000000,0.000000,False
1,439.457,0.000,0.000000,0.000000,False
2,439.470,0.013,0.000000,0.000000,False
3,439.470,0.000,0.000000,0.000000,False
4,439.470,0.000,0.000000,0.000000,False
...,...,...,...,...,...
12274,586.625,0.000,63.646096,47.221321,False
12275,586.625,0.000,63.436247,46.319841,False
12276,586.625,0.000,63.185702,45.470169,False
12277,586.625,0.000,62.899148,44.667508,False


In [20]:
dataset_pros=d3.copy()
y = dataset_pros['alarm'].astype(int)
X = dataset_pros.drop(columns=['alarm'])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
clf = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=5,
    ccp_alpha=0.01 
)

clf.fit(X_train, y_train)
print(f"Accuracy: {clf.score(X_test, y_test)}")

Accuracy: 1.0


In [ ]:
y = dataset_temp1['alarm_itog'].astype(int)
X = dataset_temp1.drop(columns=['alarm_itog'])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
clf = DecisionTreeClassifier(max_depth=5)
clf.fit(X_train, y_train)


Accuracy: 0.9857491856677525


In [21]:
code_c = m2c.export_to_c(clf)

with open("pros_model2.cpp", "w") as f:
    f.write(code_c)